# 04 mlp probes all properties

This notebook is an anonymized, output-stripped copy prepared for the GitHub study export. Paths are repo-relative; rerun cells after placing the required data and checkpoints.


# Step 3 Confounds MLP Probes For Left-Out NAT Properties

This notebook mirrors `step3_confounds_mlp_probe.ipynb`, but trains the raw and residual nonlinear probes only for the NAT Step 3 properties left out by that notebook's default priority target set. It also adds `SA-score` with the same scorer-loading logic used by the AR notebook when the scorer is available, saves a reusable SA-score panel and augmented Step 3 panel, and drops `ExactMolWt` because `MolWt` is the retained mass target.

## 1) Setup

In [ ]:
from pathlib import Path
import copy
import json
import random
import warnings
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED = 42
BATCH_SIZE = 8192
MAX_EPOCHS = 15
MIN_EPOCHS = 6
PATIENCE = 4
MIN_VAL_R2_IMPROVEMENT = 1e-3
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_WIDTH = 256
EXPECTED_SPLIT_SIZES = {'train': 635522, 'val': 79440, 'test': 79441}
RIDGE_ALPHAS = np.logspace(-3, 3, 13)
CONFOUND_COLUMNS = ['selfies_len_tokens', 'branch_token_count', 'ring_token_count', 'token_entropy']
ADDITIONAL_PROPERTIES = ['SA-score']
REDUNDANT_PROPERTIES = ['ExactMolWt']

EXCLUDED_CONFOUNDED_PROPERTIES = [
    'HeavyAtomCount',
    'MolWt',
    'ExactMolWt',
    'BertzCT',
    'RingCount',
]

STEP3_PRIORITY_PROPERTIES = [
    'cLogP',
    'TPSA',
    'FractionCSP3',
    'HBA',
    'HBD',
    'AromaticRingCount',
    'QED',
    'NumSpiroAtoms',
    'NumBridgeheadAtoms',
]

MANUAL_TARGET_PROPERTIES = None
# Example override:
# MANUAL_TARGET_PROPERTIES = ['MolWt', 'SA-score']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = bool(torch.cuda.is_available())
print('device =', device)
print('seed =', SEED)
print('batch_size =', BATCH_SIZE)
print('max_epochs =', MAX_EPOCHS)
print('use_amp =', USE_AMP)

## 2) Load Existing NAT Step 3 Artifacts

In [ ]:
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != '_patched_notebooks':
    NOTEBOOK_DIR = Path('artifacts/model_compare/nat_model_h256_l256/_patched_notebooks').resolve()

MODEL_DIR = NOTEBOOK_DIR.parent
STEP3_DIR = MODEL_DIR / 'confounds_step3'
SOURCE_MLP_DIR = MODEL_DIR / 'confounds_mlp_step3'
OUTPUT_DIR = MODEL_DIR / 'confounds_mlp_step3_missing_properties'
TABLES_DIR = OUTPUT_DIR / 'tables'
PLOTS_DIR = OUTPUT_DIR / 'plots'
MODEL_EXPORT_DIR = OUTPUT_DIR / 'mlp_models'
for path in [OUTPUT_DIR, TABLES_DIR, PLOTS_DIR, MODEL_EXPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

required_paths = {
    'panel': STEP3_DIR / 'panels_step3_with_residuals.csv',
    'z_mu': STEP3_DIR / 'Z_mu.npy',
    'ridge_raw': STEP3_DIR / 'tables' / 'r2_Z_to_Y.csv',
    'ridge_resid': STEP3_DIR / 'tables' / 'r2_Z_to_Y_vs_Yresid.csv',
    'r2_z_to_c': STEP3_DIR / 'tables' / 'r2_Z_to_C.csv',
    'top20_confounded': STEP3_DIR / 'tables' / 'top20_confounded_pairs_YC.csv',
    'corr_spearman': STEP3_DIR / 'tables' / 'corr_spearman_YC.csv',
    'corr_pearson': STEP3_DIR / 'tables' / 'corr_pearson_YC.csv',
}
missing_paths = {name: path for name, path in required_paths.items() if not path.exists()}
if missing_paths:
    msg = ['Missing required NAT Step 3 artifacts:']
    msg.extend(f'  {name}: {path}' for name, path in missing_paths.items())
    raise FileNotFoundError('\n'.join(msg))

panel = pd.read_csv(required_paths['panel'])
z_mu = np.load(required_paths['z_mu']).astype(np.float32)
step3_raw = pd.read_csv(required_paths['ridge_raw'])
step3_resid = pd.read_csv(required_paths['ridge_resid'])
r2_z_to_c = pd.read_csv(required_paths['r2_z_to_c'])
top20_confounded = pd.read_csv(required_paths['top20_confounded'])
corr_spearman = pd.read_csv(required_paths['corr_spearman'], index_col=0)
corr_pearson = pd.read_csv(required_paths['corr_pearson'], index_col=0)

if len(panel) != len(z_mu):
    raise ValueError(f'panel rows {len(panel)} != latent rows {len(z_mu)}')

print('model_dir =', MODEL_DIR)
print('step3_dir =', STEP3_DIR)
print('output_dir =', OUTPUT_DIR)
print('panel rows =', len(panel))
print('latent shape =', z_mu.shape)

## 3) Add Optional SA-score Using The AR Notebook Logic

In [ ]:
def load_sascorer():
    try:
        import sascorer
        return sascorer, None
    except Exception as first_exc:
        try:
            from rdkit.Contrib.SA_Score import sascorer
            return sascorer, None
        except Exception as second_exc:
            return None, f'SA-score unavailable: {first_exc}; {second_exc}'

def fit_ridge_step3_style(X_all, y_all, split_arr, alphas=RIDGE_ALPHAS):
    X_all = np.asarray(X_all, dtype=np.float32)
    y_all = np.asarray(y_all, dtype=float)
    split_arr = np.asarray(split_arr).astype(str)
    finite = np.isfinite(y_all) & np.isfinite(X_all).all(axis=1)
    masks = {name: (split_arr == name) & finite for name in ['train', 'val', 'test']}
    if masks['train'].sum() < 5 or np.nanstd(y_all[masks['train']]) < 1e-12:
        return {'r2_val': np.nan, 'r2_test': np.nan, 'alpha': np.nan, 'coef': None}

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    Xtr_s = x_scaler.fit_transform(X_all[masks['train']])
    ytr_s = y_scaler.fit_transform(y_all[masks['train']].reshape(-1, 1)).ravel()
    model = RidgeCV(alphas=alphas)
    model.fit(Xtr_s, ytr_s)

    def score(mask):
        if mask.sum() < 2 or np.nanstd(y_all[mask]) < 1e-12:
            return np.nan
        pred_s = model.predict(x_scaler.transform(X_all[mask]))
        pred = y_scaler.inverse_transform(pred_s.reshape(-1, 1)).ravel()
        return float(r2_score(y_all[mask], pred))

    safe_x_scale = np.where(x_scaler.scale_ == 0, 1.0, x_scaler.scale_)
    coef_original_x_space = (y_scaler.scale_[0] * model.coef_) / safe_x_scale
    return {
        'r2_val': score(masks['val']),
        'r2_test': score(masks['test']),
        'alpha': float(model.alpha_),
        'coef': coef_original_x_space.astype(np.float32),
    }

def residualize_single_target_against_confounds(panel_df, target_col):
    C_all = panel_df[CONFOUND_COLUMNS].to_numpy(dtype=float)
    y_all = pd.to_numeric(panel_df[target_col], errors='coerce').to_numpy(dtype=float)
    split_arr = panel_df['split'].astype(str).str.lower().to_numpy()
    finite = np.isfinite(y_all) & np.isfinite(C_all).all(axis=1)
    train_finite = finite & (split_arr == 'train')
    if train_finite.sum() < 10:
        raise RuntimeError(f'Not enough finite rows to residualize {target_col}')

    c_scaler = StandardScaler()
    y_scaler = StandardScaler()
    C_train_s = c_scaler.fit_transform(C_all[train_finite])
    y_train_s = y_scaler.fit_transform(y_all[train_finite].reshape(-1, 1)).ravel()
    model = RidgeCV(alphas=RIDGE_ALPHAS)
    model.fit(C_train_s, y_train_s)

    y_hat = np.full(len(y_all), np.nan, dtype=float)
    y_hat[finite] = y_scaler.inverse_transform(
        model.predict(c_scaler.transform(C_all[finite])).reshape(-1, 1)
    ).ravel()
    residual = y_all - y_hat

    def score(name):
        mask = finite & (split_arr == name)
        if mask.sum() < 2 or np.nanstd(y_all[mask]) < 1e-12:
            return np.nan
        return float(r2_score(y_all[mask], y_hat[mask]))

    return residual, {'r2_val': score('val'), 'r2_test': score('test'), 'alpha': float(model.alpha_)}

sa_score_available = False
if 'SA-score' not in panel.columns:
    sascorer, sa_warning = load_sascorer()
    if sascorer is None:
        warnings.warn(sa_warning)
    elif 'smiles' not in panel.columns:
        warnings.warn('SA-score requested, but the Step 3 panel has no smiles column for RDKit parsing.')
    else:
        from rdkit import Chem
        sa_values = []
        for smi in tqdm(panel['smiles'], desc='SA-score'):
            mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
            sa_values.append(float(sascorer.calculateScore(mol)) if mol is not None else np.nan)
        panel['SA-score'] = np.asarray(sa_values, dtype=float)

if 'SA-score' in panel.columns:
    sa_score_available = bool(pd.to_numeric(panel['SA-score'], errors='coerce').notna().sum() > 100)

if sa_score_available and 'resid_SA-score' not in panel.columns:
    panel['resid_SA-score'], sa_c_scores = residualize_single_target_against_confounds(panel, 'SA-score')
else:
    sa_c_scores = None

if sa_score_available and 'SA-score' not in set(step3_raw['property'].astype(str)):
    split_arr = panel['split'].astype(str).str.lower().to_numpy()
    y_sa = pd.to_numeric(panel['SA-score'], errors='coerce').to_numpy(dtype=float)
    raw_scores = fit_ridge_step3_style(z_mu, y_sa, split_arr)
    step3_raw = pd.concat([
        step3_raw,
        pd.DataFrame([{'property': 'SA-score', 'r2_val': raw_scores['r2_val'], 'r2_test': raw_scores['r2_test'], 'alpha': raw_scores['alpha']}]),
    ], ignore_index=True)

if sa_score_available and 'SA-score' not in set(step3_resid['property'].astype(str)):
    split_arr = panel['split'].astype(str).str.lower().to_numpy()
    y_resid_sa = pd.to_numeric(panel['resid_SA-score'], errors='coerce').to_numpy(dtype=float)
    raw_lookup = step3_raw.set_index('property').loc['SA-score']
    resid_scores = fit_ridge_step3_style(z_mu, y_resid_sa, split_arr)
    step3_resid = pd.concat([
        step3_resid,
        pd.DataFrame([{
            'property': 'SA-score',
            'r2_original': float(raw_lookup['r2_val']),
            'r2_residual': resid_scores['r2_val'],
            'delta': resid_scores['r2_val'] - float(raw_lookup['r2_val']),
            'r2_original_test': float(raw_lookup['r2_test']),
            'r2_residual_test': resid_scores['r2_test'],
            'delta_test': resid_scores['r2_test'] - float(raw_lookup['r2_test']),
        }]),
    ], ignore_index=True)

if sa_score_available and 'SA-score' not in corr_spearman.index:
    for conf in CONFOUND_COLUMNS:
        corr_spearman.loc['SA-score', conf] = pd.to_numeric(panel['SA-score'], errors='coerce').corr(
            pd.to_numeric(panel[conf], errors='coerce'), method='spearman'
        )
        corr_pearson.loc['SA-score', conf] = pd.to_numeric(panel['SA-score'], errors='coerce').corr(
            pd.to_numeric(panel[conf], errors='coerce'), method='pearson'
        )

sa_score_panel_paths = {}
augmented_panel_paths = {}
if sa_score_available:
    sa_panel_cols = [
        col for col in ['smiles', 'selfies', 'split', *CONFOUND_COLUMNS, 'SA-score', 'resid_SA-score']
        if col in panel.columns
    ]
    sa_panel = panel[sa_panel_cols].copy()
    sa_score_panel_paths = {
        'missing_mlp_tables': TABLES_DIR / 'sa_score_panel.csv',
        'step3_tables': STEP3_DIR / 'tables' / 'sa_score_panel.csv',
    }
    for path in sa_score_panel_paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)
        sa_panel.to_csv(path, index=False)

    augmented_panel_paths = {
        'step3_tables': STEP3_DIR / 'tables' / 'panels_step3_with_residuals_plus_sa.csv',
    }
    for path in augmented_panel_paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)
        panel.to_csv(path, index=False)

step3_raw.to_csv(TABLES_DIR / 'r2_Z_to_Y_with_optional_sa.csv', index=False)
step3_resid.to_csv(TABLES_DIR / 'r2_Z_to_Y_vs_Yresid_with_optional_sa.csv', index=False)
corr_spearman.to_csv(TABLES_DIR / 'corr_spearman_YC_with_optional_sa.csv')
corr_pearson.to_csv(TABLES_DIR / 'corr_pearson_YC_with_optional_sa.csv')

print('SA-score available =', sa_score_available)
if sa_score_available:
    print('SA-score finite rows =', int(pd.to_numeric(panel['SA-score'], errors='coerce').notna().sum()))
    print('SA-score panel paths =', {k: str(v) for k, v in sa_score_panel_paths.items()})
    print('Augmented panel paths =', {k: str(v) for k, v in augmented_panel_paths.items()})
    display(step3_raw[step3_raw['property'].eq('SA-score')])
    display(step3_resid[step3_resid['property'].eq('SA-score')])

## 4) Identify Left-Out Properties And Drop ExactMolWt

In [ ]:
step3_properties = step3_raw['property'].astype(str).tolist()
default_priority_targets = [
    prop for prop in STEP3_PRIORITY_PROPERTIES
    if prop in step3_properties and prop not in EXCLUDED_CONFOUNDED_PROPERTIES
]
left_out_properties = [
    prop for prop in step3_properties
    if prop not in default_priority_targets and prop not in REDUNDANT_PROPERTIES
]

if MANUAL_TARGET_PROPERTIES is None:
    TARGET_PROPERTIES = left_out_properties
else:
    TARGET_PROPERTIES = list(MANUAL_TARGET_PROPERTIES)

missing_raw_cols = [prop for prop in TARGET_PROPERTIES if prop not in panel.columns]
missing_resid_cols = [f'resid_{prop}' for prop in TARGET_PROPERTIES if f'resid_{prop}' not in panel.columns]
if missing_raw_cols or missing_resid_cols:
    raise KeyError(
        'The saved NAT Step 3 panel is missing required target columns. '
        f'missing_raw={missing_raw_cols}, missing_residual={missing_resid_cols}'
    )

existing_raw_path = SOURCE_MLP_DIR / 'tables' / 'r2_Z_to_Y_mlp.csv'
existing_resid_path = SOURCE_MLP_DIR / 'tables' / 'r2_Z_to_Yresid_mlp.csv'
existing_raw = pd.read_csv(existing_raw_path) if existing_raw_path.exists() else pd.DataFrame()
existing_resid = pd.read_csv(existing_resid_path) if existing_resid_path.exists() else pd.DataFrame()
existing_raw_props = set(existing_raw.get('target', pd.Series(dtype=str)).astype(str).str.replace('^resid_', '', regex=True)) if len(existing_raw) else set()
existing_resid_props = set(existing_resid.get('property', pd.Series(dtype=str)).astype(str)) if len(existing_resid) else set()

selection_summary = pd.DataFrame({
    'property': step3_properties,
    'default_priority_target': [p in default_priority_targets for p in step3_properties],
    'selected_for_this_notebook': [p in TARGET_PROPERTIES for p in step3_properties],
    'excluded_as_confounded_in_source': [p in EXCLUDED_CONFOUNDED_PROPERTIES for p in step3_properties],
    'dropped_as_redundant': [p in REDUNDANT_PROPERTIES for p in step3_properties],
    'additional_property_added_here': [p in ADDITIONAL_PROPERTIES for p in step3_properties],
    'already_in_existing_raw_mlp_csv': [p in existing_raw_props for p in step3_properties],
    'already_in_existing_resid_mlp_csv': [p in existing_resid_props for p in step3_properties],
})
selection_summary.to_csv(TABLES_DIR / 'missing_property_selection_summary.csv', index=False)

print('Step 3 properties:', step3_properties)
print('Default priority targets from source notebook:', default_priority_targets)
print('Dropped redundant properties:', REDUNDANT_PROPERTIES)
print('Left-out properties selected here:', TARGET_PROPERTIES)
display(selection_summary)

## 5) Recreate The Step 3 Split And Scaling

In [ ]:
split = panel['split'].astype(str).str.lower().to_numpy()
split_sizes = {name: int((split == name).sum()) for name in ('train', 'val', 'test')}
if split_sizes != EXPECTED_SPLIT_SIZES:
    raise AssertionError(split_sizes)

if 'is_rdkit_valid' in panel.columns:
    valid_mask = panel['is_rdkit_valid'].fillna(False).to_numpy(dtype=bool)
else:
    valid_mask = np.ones(len(panel), dtype=bool)

train_mask = (split == 'train') & valid_mask
val_mask = (split == 'val') & valid_mask
test_mask = (split == 'test') & valid_mask

z_mean = z_mu[train_mask].mean(axis=0).astype(np.float32)
z_std = z_mu[train_mask].std(axis=0).astype(np.float32)
z_std = np.where(z_std < 1e-8, 1.0, z_std).astype(np.float32)

X_train = ((z_mu[train_mask] - z_mean) / z_std).astype(np.float32)
X_val = ((z_mu[val_mask] - z_mean) / z_std).astype(np.float32)
X_test = ((z_mu[test_mask] - z_mean) / z_std).astype(np.float32)

panel_train = panel.loc[train_mask].reset_index(drop=True)
panel_val = panel.loc[val_mask].reset_index(drop=True)
panel_test = panel.loc[test_mask].reset_index(drop=True)

print('split sizes =', split_sizes)
print('valid train/val/test =', int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))
print('latent dim =', X_train.shape[1])

## 6) MLP Helper Functions

In [ ]:
RAW_MODEL_REGISTRY = {}
RESID_MODEL_REGISTRY = {}

def safe_r2(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 2:
        return np.nan
    if np.std(y_true[mask]) < 1e-12:
        return np.nan
    return float(r2_score(y_true[mask], y_pred[mask]))

def train_target_stats(y_train_full):
    finite = np.isfinite(y_train_full)
    if finite.sum() == 0:
        return 0.0, 1.0
    mean = float(np.nanmean(y_train_full[finite]))
    std = float(np.nanstd(y_train_full[finite]))
    if not np.isfinite(std) or std < 1e-8:
        std = 1.0
    return mean, std

class MLPProbe(torch.nn.Module):
    def __init__(self, input_dim, hidden_width=HIDDEN_WIDTH):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_width),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_width, hidden_width),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_width, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def predict_raw(model, x_np, y_mean, y_std, batch_size=BATCH_SIZE):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(x_np), batch_size):
            stop = min(start + batch_size, len(x_np))
            xb = torch.from_numpy(x_np[start:stop]).to(device, non_blocking=True)
            pred_std = model(xb)
            pred_raw = pred_std * y_std + y_mean
            outputs.append(pred_raw.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)

def fit_mlp_target(target_name, y_train_full, y_val_full, y_test_full, registry=None):
    train_finite = np.isfinite(y_train_full)
    val_finite = np.isfinite(y_val_full)
    test_finite = np.isfinite(y_test_full)

    xtr = X_train[train_finite]
    xva = X_val[val_finite]
    xte = X_test[test_finite]

    ytr_raw = y_train_full[train_finite].astype(np.float32)
    yva_raw = y_val_full[val_finite].astype(np.float32)
    yte_raw = y_test_full[test_finite].astype(np.float32)

    y_mean, y_std = train_target_stats(y_train_full)
    ytr_std = ((ytr_raw - y_mean) / y_std).astype(np.float32)

    dataset = torch.utils.data.TensorDataset(torch.from_numpy(xtr), torch.from_numpy(ytr_std))
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
    )

    model = MLPProbe(X_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_state = None
    best_val_r2 = -np.inf
    best_epoch = -1
    bad_epochs = 0
    t0 = perf_counter()

    for epoch in range(MAX_EPOCHS):
        model.train()
        running_loss = 0.0
        sample_count = 0
        batch_iter = tqdm(loader, total=len(loader), leave=False, mininterval=0.5, desc=f'{target_name} | epoch {epoch + 1}/{MAX_EPOCHS}')
        for xb, yb in batch_iter:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred = model(xb)
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            batch_n = int(yb.shape[0])
            running_loss += float(loss.detach().cpu()) * batch_n
            sample_count += batch_n
            batch_iter.set_postfix(loss=f'{float(loss.detach().cpu()):.4f}')

        val_pred = predict_raw(model, xva, y_mean, y_std)
        val_r2 = safe_r2(yva_raw, val_pred)
        mean_train_loss = running_loss / max(sample_count, 1)
        print(f'epoch {epoch + 1:02d} | {target_name} | train_loss={mean_train_loss:.4f} | val_r2={val_r2:.4f}')

        improved = np.isfinite(val_r2) and (val_r2 > best_val_r2 + MIN_VAL_R2_IMPROVEMENT)
        if improved:
            best_val_r2 = val_r2
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch + 1 >= MIN_EPOCHS and bad_epochs >= PATIENCE:
            print(f'early stop on {target_name} at epoch {epoch + 1}')
            break

    if best_state is None:
        raise RuntimeError(f'No valid model state for {target_name}')

    model.load_state_dict(best_state)
    train_pred = predict_raw(model, xtr, y_mean, y_std)
    val_pred = predict_raw(model, xva, y_mean, y_std)
    test_pred = predict_raw(model, xte, y_mean, y_std)
    elapsed_seconds = perf_counter() - t0

    result = {
        'target': target_name,
        'best_epoch': int(best_epoch + 1),
        'elapsed_seconds': float(elapsed_seconds),
        'y_mean_train': float(y_mean),
        'y_std_train': float(y_std),
        'n_train': int(train_finite.sum()),
        'n_val': int(val_finite.sum()),
        'n_test': int(test_finite.sum()),
        'r2_train': safe_r2(ytr_raw, train_pred),
        'r2_val': safe_r2(yva_raw, val_pred),
        'r2_test': safe_r2(yte_raw, test_pred),
    }

    if registry is not None:
        registry[str(target_name)] = {
            'state_dict': {k: v.detach().cpu().clone() for k, v in best_state.items()},
            'input_dim': int(X_train.shape[1]),
            'hidden_width': int(HIDDEN_WIDTH),
            'target_name': str(target_name),
            'y_mean_train': float(y_mean),
            'y_std_train': float(y_std),
            'best_epoch': int(result['best_epoch']),
            'r2_val': float(result['r2_val']),
            'r2_test': float(result['r2_test']),
        }
    print(f"finished {target_name}: best_epoch={result['best_epoch']}, val_r2={result['r2_val']:.4f}, test_r2={result['r2_test']:.4f}, seconds={result['elapsed_seconds']:.1f}")
    return result

## 7) Fit MLP On Raw Left-Out Properties

In [ ]:
raw_rows = []
for prop in TARGET_PROPERTIES:
    print('\n' + '=' * 80)
    print(f'starting raw target: {prop}')
    raw_rows.append(
        fit_mlp_target(
            prop,
            pd.to_numeric(panel_train[prop], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_val[prop], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_test[prop], errors='coerce').to_numpy(dtype=float),
            registry=RAW_MODEL_REGISTRY,
        )
    )

mlp_raw_df = pd.DataFrame(raw_rows).sort_values('r2_test', ascending=False).reset_index(drop=True)
mlp_raw_df['property'] = mlp_raw_df['target']
mlp_raw_df.to_csv(TABLES_DIR / 'r2_Z_to_Y_missing_mlp.csv', index=False)
display(mlp_raw_df)

## 8) Fit MLP On Residual Left-Out Properties

In [ ]:
resid_rows = []
for prop in TARGET_PROPERTIES:
    target_name = f'resid_{prop}'
    print('\n' + '=' * 80)
    print(f'starting residual target: {target_name}')
    resid_rows.append(
        fit_mlp_target(
            target_name,
            pd.to_numeric(panel_train[target_name], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_val[target_name], errors='coerce').to_numpy(dtype=float),
            pd.to_numeric(panel_test[target_name], errors='coerce').to_numpy(dtype=float),
            registry=RESID_MODEL_REGISTRY,
        )
    )

mlp_resid_df = pd.DataFrame(resid_rows)
mlp_resid_df['property'] = mlp_resid_df['target'].str.replace('^resid_', '', regex=True)
mlp_resid_df = mlp_resid_df.sort_values('r2_test', ascending=False).reset_index(drop=True)
mlp_resid_df.to_csv(TABLES_DIR / 'r2_Z_to_Yresid_missing_mlp.csv', index=False)
display(mlp_resid_df)

## 9) Compare MLP To Step 3 Ridge

In [ ]:
raw_compare = mlp_raw_df.merge(
    step3_raw[['property', 'r2_val', 'r2_test']].rename(columns={'r2_val': 'ridge_r2_val', 'r2_test': 'ridge_r2_test'}),
    on='property',
    how='left',
)
raw_compare['delta_val_mlp_minus_ridge'] = raw_compare['r2_val'] - raw_compare['ridge_r2_val']
raw_compare['delta_test_mlp_minus_ridge'] = raw_compare['r2_test'] - raw_compare['ridge_r2_test']
raw_compare.to_csv(TABLES_DIR / 'mlp_vs_ridge_missing_raw.csv', index=False)

resid_compare = mlp_resid_df.merge(
    step3_resid[['property', 'r2_residual', 'r2_residual_test', 'delta_test']].rename(columns={'r2_residual': 'ridge_r2_val', 'r2_residual_test': 'ridge_r2_test'}),
    on='property',
    how='left',
)
resid_compare['delta_val_mlp_minus_ridge'] = resid_compare['r2_val'] - resid_compare['ridge_r2_val']
resid_compare['delta_test_mlp_minus_ridge'] = resid_compare['r2_test'] - resid_compare['ridge_r2_test']
resid_compare.to_csv(TABLES_DIR / 'mlp_vs_ridge_missing_resid.csv', index=False)

raw_vs_resid = (
    mlp_raw_df[['property', 'r2_val', 'r2_test']]
    .rename(columns={'r2_val': 'mlp_raw_r2_val', 'r2_test': 'mlp_raw_r2_test'})
    .merge(
        mlp_resid_df[['property', 'r2_val', 'r2_test']].rename(columns={'r2_val': 'mlp_resid_r2_val', 'r2_test': 'mlp_resid_r2_test'}),
        on='property',
        how='left',
    )
)
raw_vs_resid['delta_test_resid_minus_raw'] = raw_vs_resid['mlp_resid_r2_test'] - raw_vs_resid['mlp_raw_r2_test']
raw_vs_resid.to_csv(TABLES_DIR / 'mlp_missing_raw_vs_resid.csv', index=False)

display(raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False))
display(resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False))
display(raw_vs_resid.sort_values('delta_test_resid_minus_raw', ascending=False))

## 10) Quick Visual Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

top_raw = raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False)
axes[0].barh(top_raw['property'], top_raw['delta_test_mlp_minus_ridge'])
axes[0].invert_yaxis()
axes[0].set_title('Raw: MLP - Ridge (test R^2)')

top_resid = resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False)
axes[1].barh(top_resid['target'], top_resid['delta_test_mlp_minus_ridge'])
axes[1].invert_yaxis()
axes[1].set_title('Residual: MLP - Ridge (test R^2)')

gap = raw_vs_resid.sort_values('delta_test_resid_minus_raw', ascending=False)
axes[2].barh(gap['property'], gap['delta_test_resid_minus_raw'])
axes[2].invert_yaxis()
axes[2].set_title('MLP: Residual - Raw (test R^2)')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'mlp_missing_probe_summary.png', dpi=220)
plt.show()

## 11) Combined Delta R2 Table

In [ ]:
max_abs_spearman = corr_spearman.abs().max(axis=1).rename('max_abs_spearman').reset_index().rename(columns={'index': 'property'})

delta_r2_df = (
    raw_compare[['property', 'r2_test', 'ridge_r2_test', 'delta_test_mlp_minus_ridge']]
    .rename(columns={
        'r2_test': 'raw_mlp_r2_test',
        'ridge_r2_test': 'raw_ridge_r2_test',
        'delta_test_mlp_minus_ridge': 'raw_delta_r2_mlp_minus_ridge',
    })
    .merge(
        resid_compare[['property', 'target', 'r2_test', 'ridge_r2_test', 'delta_test_mlp_minus_ridge']]
        .rename(columns={
            'target': 'resid_target',
            'r2_test': 'resid_mlp_r2_test',
            'ridge_r2_test': 'resid_ridge_r2_test',
            'delta_test_mlp_minus_ridge': 'resid_delta_r2_mlp_minus_ridge',
        }),
        on='property',
        how='left',
    )
    .merge(max_abs_spearman, on='property', how='left')
)

delta_r2_df['resid_minus_raw_delta_r2'] = (
    delta_r2_df['resid_delta_r2_mlp_minus_ridge'] - delta_r2_df['raw_delta_r2_mlp_minus_ridge']
)
delta_r2_df = delta_r2_df.sort_values('property').reset_index(drop=True)
delta_r2_df.to_csv(TABLES_DIR / 'mlp_ridge_missing_delta_r2_comparison.csv', index=False)
display(delta_r2_df)

## 12) Save Summary And Model Weights

In [ ]:
summary = {
    'model_dir': str(MODEL_DIR),
    'step3_dir': str(STEP3_DIR),
    'source_mlp_dir': str(SOURCE_MLP_DIR),
    'output_dir': str(OUTPUT_DIR),
    'device': str(device),
    'batch_size': int(BATCH_SIZE),
    'max_epochs': int(MAX_EPOCHS),
    'min_epochs': int(MIN_EPOCHS),
    'patience': int(PATIENCE),
    'min_val_r2_improvement': float(MIN_VAL_R2_IMPROVEMENT),
    'use_amp': bool(USE_AMP),
    'target_properties': list(TARGET_PROPERTIES),
    'default_priority_targets_from_source_notebook': list(default_priority_targets),
    'excluded_confounded_properties_from_source_notebook': list(EXCLUDED_CONFOUNDED_PROPERTIES),
    'additional_properties_requested': list(ADDITIONAL_PROPERTIES),
    'sa_score_available': bool(sa_score_available),
    'sa_score_panel_paths': {k: str(v) for k, v in sa_score_panel_paths.items()},
    'augmented_panel_paths': {k: str(v) for k, v in augmented_panel_paths.items()},
    'dropped_redundant_properties': list(REDUNDANT_PROPERTIES),
    'top_raw_test_gains': raw_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False).head(5)[['property', 'delta_test_mlp_minus_ridge', 'r2_test', 'ridge_r2_test']].to_dict(orient='records'),
    'top_resid_test_gains': resid_compare.sort_values('delta_test_mlp_minus_ridge', ascending=False).head(5)[['target', 'delta_test_mlp_minus_ridge', 'r2_test', 'ridge_r2_test']].to_dict(orient='records'),
}

with open(OUTPUT_DIR / 'summary_mlp_missing_confounds.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

saved_rows = []
for variant_name, registry in [('raw', RAW_MODEL_REGISTRY), ('residual', RESID_MODEL_REGISTRY)]:
    for target_name, bundle in registry.items():
        out_path = MODEL_EXPORT_DIR / f'{target_name}.pt'
        torch.save(bundle, out_path)
        saved_rows.append({
            'variant': variant_name,
            'target_name': str(target_name),
            'path': str(out_path),
            'best_epoch': int(bundle['best_epoch']),
            'r2_val': float(bundle['r2_val']),
            'r2_test': float(bundle['r2_test']),
        })

if saved_rows:
    checkpoint_manifest = pd.DataFrame(saved_rows).sort_values(['variant', 'target_name']).reset_index(drop=True)
    checkpoint_manifest.to_csv(TABLES_DIR / 'mlp_missing_checkpoint_manifest.csv', index=False)
    display(checkpoint_manifest)
else:
    print('No checkpoints were exported.')

display(pd.DataFrame([summary]))